# Cypher: Relationships and Path Queries

## Overview
Relationships are first-class citizens in Cypher. This notebook covers multi-hop traversals, variable-length paths, OPTIONAL MATCH, and shortestPath — the queries that make graph databases compelling.

**Relationship pattern syntax:**
```cypher
(a)-[:TYPE]->(b)           -- directed: a to b
(a)<-[:TYPE]-(b)           -- directed: b to a
(a)-[:TYPE]-(b)            -- undirected: either direction
(a)-[:TYPE*2]->(b)         -- exactly 2 hops
(a)-[:TYPE*1..3]->(b)      -- 1 to 3 hops
(a)-[:TYPE*]->(b)          -- any number of hops (use with care)
p = (a)-[*..6]->(b)        -- assign path to variable p
```

**Path functions:**
```cypher
length(p)          -- number of relationships in path
nodes(p)           -- list of nodes
relationships(p)   -- list of relationships
```

---

In [1]:
# No Neo4j server needed for this notebook.
# We build a pure-Python in-memory graph to demonstrate concepts,
# then show the equivalent Neo4j/Cypher representation throughout.

class Node:
    def __init__(self, label, **props):
        self.label = label
        self.props = props
    def __repr__(self):
        p = ', '.join(f'{k}: {v!r}' for k, v in self.props.items())
        return f"(:{self.label} {{{p}}})"

class Relationship:
    def __init__(self, start, rel_type, end, **props):
        self.start    = start
        self.rel_type = rel_type
        self.end      = end
        self.props    = props
    def __repr__(self):
        p = ' ' + str(self.props) if self.props else ''
        return f"({self.start.props.get('name','?')})-[:{self.rel_type}{p}]->({self.end.props.get('name','?')})"

# ── Healthcare + finance graph dataset ───────────────────────────
# People, organisations, medications, conditions, accounts

alice   = Node("Person",   name="Alice Ngata",    age=45, province="NB")
bob     = Node("Person",   name="Bob Chen",       age=52, province="ON")
fatima  = Node("Person",   name="Fatima Rashid",  age=38, province="BC")
james   = Node("Person",   name="James MacLeod",  age=61, province="NS")

bcmc    = Node("Hospital", name="BCMC",           city="Vancouver")
ngh     = Node("Hospital", name="NGH",            city="Moncton")

dr_lee  = Node("Doctor",   name="Dr. Lee",        specialty="Cardiology")
dr_pham = Node("Doctor",   name="Dr. Pham",       specialty="Endocrinology")

lisinopril = Node("Drug",  name="Lisinopril",     class_="ACE inhibitor")
metformin  = Node("Drug",  name="Metformin",      class_="Biguanide")
salbutamol = Node("Drug",  name="Salbutamol",     class_="Beta-2 agonist")

hypertension = Node("Condition", name="Hypertension", icd10="I10")
diabetes     = Node("Condition", name="Type 2 Diabetes", icd10="E11")
asthma       = Node("Condition", name="Asthma",      icd10="J45")

relationships = [
    Relationship(alice,  "TREATED_AT",    bcmc,        since="2020"),
    Relationship(bob,    "TREATED_AT",    ngh,         since="2022"),
    Relationship(fatima, "TREATED_AT",    bcmc,        since="2021"),
    Relationship(james,  "TREATED_AT",    ngh,         since="2019"),
    Relationship(dr_lee, "WORKS_AT",      bcmc),
    Relationship(dr_pham,"WORKS_AT",      ngh),
    Relationship(dr_lee, "TREATS",        alice),
    Relationship(dr_lee, "TREATS",        fatima),
    Relationship(dr_pham,"TREATS",        bob),
    Relationship(dr_pham,"TREATS",        james),
    Relationship(alice,  "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", diabetes),
    Relationship(fatima, "HAS_CONDITION", asthma),
    Relationship(james,  "HAS_CONDITION", diabetes),
    Relationship(alice,  "PRESCRIBED",    lisinopril,  dose="10mg", since="2023-01"),
    Relationship(bob,    "PRESCRIBED",    lisinopril,  dose="5mg",  since="2022-06"),
    Relationship(bob,    "PRESCRIBED",    metformin,   dose="500mg",since="2022-06"),
    Relationship(james,  "PRESCRIBED",    metformin,   dose="1g",   since="2023-03"),
    Relationship(fatima, "PRESCRIBED",    salbutamol,  dose="2.5mg",since="2021-09"),
    Relationship(lisinopril, "TREATS",    hypertension),
    Relationship(metformin,  "TREATS",    diabetes),
    Relationship(salbutamol, "TREATS",    asthma),
]

all_nodes = [alice, bob, fatima, james, bcmc, ngh,
             dr_lee, dr_pham, lisinopril, metformin, salbutamol,
             hypertension, diabetes, asthma]

print(f"Graph loaded: {len(all_nodes)} nodes, {len(relationships)} relationships")
label_counts = {}
for n in all_nodes:
    label_counts[n.label] = label_counts.get(n.label, 0) + 1
for label, count in sorted(label_counts.items()):
    print(f"  :{label:<12s}  {count} nodes")


print("=== Relationship patterns and multi-hop traversal ===")
print()

# Build adjacency for traversal
def rels_from(node, rel_type=None):
    return [r for r in relationships
            if r.start is node and (rel_type is None or r.rel_type == rel_type)]

def rels_to(node, rel_type=None):
    return [r for r in relationships
            if r.end is node and (rel_type is None or r.rel_type == rel_type)]

# 1-hop: who does Dr. Lee treat?
print("1-hop: MATCH (d:Doctor {name:'Dr. Lee'})-[:TREATS]->(p:Person)")
for r in rels_from(dr_lee, 'TREATS'):
    print(f"  Dr. Lee -[:TREATS]-> {r.end.props['name']}")

print()
# 2-hop: patients of Dr. Lee and their conditions
print("2-hop: MATCH (d:Doctor)-[:TREATS]->(p:Person)-[:HAS_CONDITION]->(c:Condition)")
print("       WHERE d.name = 'Dr. Lee'")
for r1 in rels_from(dr_lee, 'TREATS'):
    patient = r1.end
    for r2 in rels_from(patient, 'HAS_CONDITION'):
        cond = r2.end
        print(f"  Dr. Lee -> {patient.props['name']:<18s} -> {cond.props['name']}")

print()
# Cypher variable-length path syntax
print("Variable-length path Cypher syntax:")
vl_examples = [
    ("(a)-[:KNOWS*2]->(b)",    "Exactly 2 hops along :KNOWS"),
    ("(a)-[:KNOWS*1..3]->(b)", "1, 2, or 3 hops along :KNOWS"),
    ("(a)-[:KNOWS*]->(b)",     "Any number of hops (use carefully — can be slow)"),
    ("(a)-[*1..3]->(b)",       "1–3 hops of ANY relationship type"),
    ("(a)-[:KNOWS*]->(b) WHERE a <> b", "All reachable nodes (exclude self)"),
]
for pattern, desc in vl_examples:
    print(f"  {pattern:<40s}  {desc}")


Graph loaded: 14 nodes, 23 relationships
  :Condition     3 nodes
  :Doctor        2 nodes
  :Drug          3 nodes
  :Hospital      2 nodes
  :Person        4 nodes
=== Relationship patterns and multi-hop traversal ===

1-hop: MATCH (d:Doctor {name:'Dr. Lee'})-[:TREATS]->(p:Person)
  Dr. Lee -[:TREATS]-> Alice Ngata
  Dr. Lee -[:TREATS]-> Fatima Rashid

2-hop: MATCH (d:Doctor)-[:TREATS]->(p:Person)-[:HAS_CONDITION]->(c:Condition)
       WHERE d.name = 'Dr. Lee'
  Dr. Lee -> Alice Ngata        -> Hypertension
  Dr. Lee -> Fatima Rashid      -> Asthma

Variable-length path Cypher syntax:
  (a)-[:KNOWS*2]->(b)                       Exactly 2 hops along :KNOWS
  (a)-[:KNOWS*1..3]->(b)                    1, 2, or 3 hops along :KNOWS
  (a)-[:KNOWS*]->(b)                        Any number of hops (use carefully — can be slow)
  (a)-[*1..3]->(b)                          1–3 hops of ANY relationship type
  (a)-[:KNOWS*]->(b) WHERE a <> b           All reachable nodes (exclude self)


---
## OPTIONAL MATCH and undirected relationships

In [2]:
# No Neo4j server needed for this notebook.
# We build a pure-Python in-memory graph to demonstrate concepts,
# then show the equivalent Neo4j/Cypher representation throughout.

class Node:
    def __init__(self, label, **props):
        self.label = label
        self.props = props
    def __repr__(self):
        p = ', '.join(f'{k}: {v!r}' for k, v in self.props.items())
        return f"(:{self.label} {{{p}}})"

class Relationship:
    def __init__(self, start, rel_type, end, **props):
        self.start    = start
        self.rel_type = rel_type
        self.end      = end
        self.props    = props
    def __repr__(self):
        p = ' ' + str(self.props) if self.props else ''
        return f"({self.start.props.get('name','?')})-[:{self.rel_type}{p}]->({self.end.props.get('name','?')})"

# ── Healthcare + finance graph dataset ───────────────────────────
# People, organisations, medications, conditions, accounts

alice   = Node("Person",   name="Alice Ngata",    age=45, province="NB")
bob     = Node("Person",   name="Bob Chen",       age=52, province="ON")
fatima  = Node("Person",   name="Fatima Rashid",  age=38, province="BC")
james   = Node("Person",   name="James MacLeod",  age=61, province="NS")

bcmc    = Node("Hospital", name="BCMC",           city="Vancouver")
ngh     = Node("Hospital", name="NGH",            city="Moncton")

dr_lee  = Node("Doctor",   name="Dr. Lee",        specialty="Cardiology")
dr_pham = Node("Doctor",   name="Dr. Pham",       specialty="Endocrinology")

lisinopril = Node("Drug",  name="Lisinopril",     class_="ACE inhibitor")
metformin  = Node("Drug",  name="Metformin",      class_="Biguanide")
salbutamol = Node("Drug",  name="Salbutamol",     class_="Beta-2 agonist")

hypertension = Node("Condition", name="Hypertension", icd10="I10")
diabetes     = Node("Condition", name="Type 2 Diabetes", icd10="E11")
asthma       = Node("Condition", name="Asthma",      icd10="J45")

relationships = [
    Relationship(alice,  "TREATED_AT",    bcmc,        since="2020"),
    Relationship(bob,    "TREATED_AT",    ngh,         since="2022"),
    Relationship(fatima, "TREATED_AT",    bcmc,        since="2021"),
    Relationship(james,  "TREATED_AT",    ngh,         since="2019"),
    Relationship(dr_lee, "WORKS_AT",      bcmc),
    Relationship(dr_pham,"WORKS_AT",      ngh),
    Relationship(dr_lee, "TREATS",        alice),
    Relationship(dr_lee, "TREATS",        fatima),
    Relationship(dr_pham,"TREATS",        bob),
    Relationship(dr_pham,"TREATS",        james),
    Relationship(alice,  "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", diabetes),
    Relationship(fatima, "HAS_CONDITION", asthma),
    Relationship(james,  "HAS_CONDITION", diabetes),
    Relationship(alice,  "PRESCRIBED",    lisinopril,  dose="10mg", since="2023-01"),
    Relationship(bob,    "PRESCRIBED",    lisinopril,  dose="5mg",  since="2022-06"),
    Relationship(bob,    "PRESCRIBED",    metformin,   dose="500mg",since="2022-06"),
    Relationship(james,  "PRESCRIBED",    metformin,   dose="1g",   since="2023-03"),
    Relationship(fatima, "PRESCRIBED",    salbutamol,  dose="2.5mg",since="2021-09"),
    Relationship(lisinopril, "TREATS",    hypertension),
    Relationship(metformin,  "TREATS",    diabetes),
    Relationship(salbutamol, "TREATS",    asthma),
]

all_nodes = [alice, bob, fatima, james, bcmc, ngh,
             dr_lee, dr_pham, lisinopril, metformin, salbutamol,
             hypertension, diabetes, asthma]

print(f"Graph loaded: {len(all_nodes)} nodes, {len(relationships)} relationships")
label_counts = {}
for n in all_nodes:
    label_counts[n.label] = label_counts.get(n.label, 0) + 1
for label, count in sorted(label_counts.items()):
    print(f"  :{label:<12s}  {count} nodes")


print("=== OPTIONAL MATCH (LEFT JOIN equivalent) ===")
print()

print("""Cypher: OPTIONAL MATCH — returns NULL for missing relationship (like LEFT JOIN)

  MATCH  (p:Person)
  OPTIONAL MATCH (p)-[:PRESCRIBED]->(d:Drug)
  RETURN p.name, d.name   -- d.name is NULL if no prescriptions

  SQL equivalent:
  SELECT pt.name, dr.name
  FROM   patients pt
  LEFT JOIN prescriptions rx ON rx.patient_id = pt.id
  LEFT JOIN drugs dr          ON dr.id = rx.drug_id;
""")

# Simulate OPTIONAL MATCH
print("Simulated result (all persons, drugs or NULL):")
prescribed_map = {}
for r in relationships:
    if r.rel_type == 'PRESCRIBED' and r.start.label == 'Person':
        prescribed_map.setdefault(r.start.props['name'], []).append(r.end.props['name'])

for p in sorted([n for n in all_nodes if n.label == 'Person'],
                key=lambda n: n.props['name']):
    drugs = prescribed_map.get(p.props['name'], [None])
    for drug in drugs:
        print(f"  {p.props['name']:<22s}  {str(drug)}")

print()
print("=== MATCH undirected relationships ===")
print("""
  -- Cypher: direction can be omitted to match either direction
  MATCH (a:Person)-[:KNOWS]-(b:Person)   -- matches a->b OR b->a
  RETURN a.name, b.name

  -- Find all nodes connected to Alice regardless of direction:
  MATCH (p:Person {name:'Alice Ngata'})--(connected)
  RETURN connected
  -- returns all nodes with ANY relationship to Alice (any type, any direction)
""")


Graph loaded: 14 nodes, 23 relationships
  :Condition     3 nodes
  :Doctor        2 nodes
  :Drug          3 nodes
  :Hospital      2 nodes
  :Person        4 nodes
=== OPTIONAL MATCH (LEFT JOIN equivalent) ===

Cypher: OPTIONAL MATCH — returns NULL for missing relationship (like LEFT JOIN)

  MATCH  (p:Person)
  OPTIONAL MATCH (p)-[:PRESCRIBED]->(d:Drug)
  RETURN p.name, d.name   -- d.name is NULL if no prescriptions

  SQL equivalent:
  SELECT pt.name, dr.name
  FROM   patients pt
  LEFT JOIN prescriptions rx ON rx.patient_id = pt.id
  LEFT JOIN drugs dr          ON dr.id = rx.drug_id;

Simulated result (all persons, drugs or NULL):
  Alice Ngata             Lisinopril
  Bob Chen                Lisinopril
  Bob Chen                Metformin
  Fatima Rashid           Salbutamol
  James MacLeod           Metformin

=== MATCH undirected relationships ===

  -- Cypher: direction can be omitted to match either direction
  MATCH (a:Person)-[:KNOWS]-(b:Person)   -- matches a->b OR b->a
  R

---
## shortestPath and BFS traversal

In [3]:
# No Neo4j server needed for this notebook.
# We build a pure-Python in-memory graph to demonstrate concepts,
# then show the equivalent Neo4j/Cypher representation throughout.

class Node:
    def __init__(self, label, **props):
        self.label = label
        self.props = props
    def __repr__(self):
        p = ', '.join(f'{k}: {v!r}' for k, v in self.props.items())
        return f"(:{self.label} {{{p}}})"

class Relationship:
    def __init__(self, start, rel_type, end, **props):
        self.start    = start
        self.rel_type = rel_type
        self.end      = end
        self.props    = props
    def __repr__(self):
        p = ' ' + str(self.props) if self.props else ''
        return f"({self.start.props.get('name','?')})-[:{self.rel_type}{p}]->({self.end.props.get('name','?')})"

# ── Healthcare + finance graph dataset ───────────────────────────
# People, organisations, medications, conditions, accounts

alice   = Node("Person",   name="Alice Ngata",    age=45, province="NB")
bob     = Node("Person",   name="Bob Chen",       age=52, province="ON")
fatima  = Node("Person",   name="Fatima Rashid",  age=38, province="BC")
james   = Node("Person",   name="James MacLeod",  age=61, province="NS")

bcmc    = Node("Hospital", name="BCMC",           city="Vancouver")
ngh     = Node("Hospital", name="NGH",            city="Moncton")

dr_lee  = Node("Doctor",   name="Dr. Lee",        specialty="Cardiology")
dr_pham = Node("Doctor",   name="Dr. Pham",       specialty="Endocrinology")

lisinopril = Node("Drug",  name="Lisinopril",     class_="ACE inhibitor")
metformin  = Node("Drug",  name="Metformin",      class_="Biguanide")
salbutamol = Node("Drug",  name="Salbutamol",     class_="Beta-2 agonist")

hypertension = Node("Condition", name="Hypertension", icd10="I10")
diabetes     = Node("Condition", name="Type 2 Diabetes", icd10="E11")
asthma       = Node("Condition", name="Asthma",      icd10="J45")

relationships = [
    Relationship(alice,  "TREATED_AT",    bcmc,        since="2020"),
    Relationship(bob,    "TREATED_AT",    ngh,         since="2022"),
    Relationship(fatima, "TREATED_AT",    bcmc,        since="2021"),
    Relationship(james,  "TREATED_AT",    ngh,         since="2019"),
    Relationship(dr_lee, "WORKS_AT",      bcmc),
    Relationship(dr_pham,"WORKS_AT",      ngh),
    Relationship(dr_lee, "TREATS",        alice),
    Relationship(dr_lee, "TREATS",        fatima),
    Relationship(dr_pham,"TREATS",        bob),
    Relationship(dr_pham,"TREATS",        james),
    Relationship(alice,  "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", hypertension),
    Relationship(bob,    "HAS_CONDITION", diabetes),
    Relationship(fatima, "HAS_CONDITION", asthma),
    Relationship(james,  "HAS_CONDITION", diabetes),
    Relationship(alice,  "PRESCRIBED",    lisinopril,  dose="10mg", since="2023-01"),
    Relationship(bob,    "PRESCRIBED",    lisinopril,  dose="5mg",  since="2022-06"),
    Relationship(bob,    "PRESCRIBED",    metformin,   dose="500mg",since="2022-06"),
    Relationship(james,  "PRESCRIBED",    metformin,   dose="1g",   since="2023-03"),
    Relationship(fatima, "PRESCRIBED",    salbutamol,  dose="2.5mg",since="2021-09"),
    Relationship(lisinopril, "TREATS",    hypertension),
    Relationship(metformin,  "TREATS",    diabetes),
    Relationship(salbutamol, "TREATS",    asthma),
]

all_nodes = [alice, bob, fatima, james, bcmc, ngh,
             dr_lee, dr_pham, lisinopril, metformin, salbutamol,
             hypertension, diabetes, asthma]

print(f"Graph loaded: {len(all_nodes)} nodes, {len(relationships)} relationships")
label_counts = {}
for n in all_nodes:
    label_counts[n.label] = label_counts.get(n.label, 0) + 1
for label, count in sorted(label_counts.items()):
    print(f"  :{label:<12s}  {count} nodes")


print("=== Path queries: shortestPath and allShortestPaths ===")
print()

print("""Cypher shortest path:

  -- Shortest path between Alice and any node prescribed Metformin:
  MATCH  (alice:Person {name: 'Alice Ngata'}),
         (met:Drug     {name: 'Metformin'})
  MATCH  p = shortestPath((alice)-[*..6]->(met))
  RETURN p, length(p) AS hops

  -- All shortest paths:
  MATCH  p = allShortestPaths(
      (a:Person {name:'Alice Ngata'})-[*]-(b:Person {name:'Bob Chen'})
  )
  RETURN p
""")

# Simulate BFS shortest path in Python
def bfs_path(start_node, end_name, all_rels):
    from collections import deque
    queue = deque([[start_node]])
    visited = {id(start_node)}
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node.props.get('name') == end_name:
            return path
        for r in all_rels:
            nxt = None
            if r.start is node and id(r.end) not in visited:
                nxt = r.end
            elif r.end is node and id(r.start) not in visited:
                nxt = r.start
            if nxt:
                visited.add(id(nxt))
                queue.append(path + [nxt])
    return None

print("Simulated BFS paths from Alice:")
for target in ['Bob Chen', 'Metformin', 'Type 2 Diabetes']:
    path = bfs_path(alice, target, relationships)
    if path:
        names = [n.props.get('name', n.label) for n in path]
        print(f"  Alice -> {target}: {' -> '.join(names)}  ({len(path)-1} hops)")

print()
print("Key path functions in Cypher:")
path_fns = [
    ("shortestPath((a)-[*..N]->(b))", "Single shortest path; N = max hop limit"),
    ("allShortestPaths((a)-[*]-(b))", "All paths of minimum length"),
    ("length(p)",                     "Number of relationships in path p"),
    ("nodes(p)",                      "List of all nodes in path"),
    ("relationships(p)",              "List of all relationships in path"),
    ("p[0]",                          "First node in path"),
]
for fn, desc in path_fns:
    print(f"  {fn:<42s}  {desc}")


Graph loaded: 14 nodes, 23 relationships
  :Condition     3 nodes
  :Doctor        2 nodes
  :Drug          3 nodes
  :Hospital      2 nodes
  :Person        4 nodes
=== Path queries: shortestPath and allShortestPaths ===

Cypher shortest path:

  -- Shortest path between Alice and any node prescribed Metformin:
  MATCH  (alice:Person {name: 'Alice Ngata'}),
         (met:Drug     {name: 'Metformin'})
  MATCH  p = shortestPath((alice)-[*..6]->(met))
  RETURN p, length(p) AS hops

  -- All shortest paths:
  MATCH  p = allShortestPaths(
      (a:Person {name:'Alice Ngata'})-[*]-(b:Person {name:'Bob Chen'})
  )
  RETURN p

Simulated BFS paths from Alice:
  Alice -> Bob Chen: Alice Ngata -> Hypertension -> Bob Chen  (2 hops)
  Alice -> Metformin: Alice Ngata -> Hypertension -> Bob Chen -> Metformin  (3 hops)
  Alice -> Type 2 Diabetes: Alice Ngata -> Hypertension -> Bob Chen -> Type 2 Diabetes  (3 hops)

Key path functions in Cypher:
  shortestPath((a)-[*..N]->(b))               Single sho

---
## Common Pitfalls

**1. Unbounded variable-length paths causing query explosion**
`MATCH (a)-[*]->(b)` with no hop limit on a dense graph can generate millions of paths and run indefinitely. Always set a maximum hop count: `(a)-[*..6]->(b)`. Use `shortestPath()` rather than enumerating all paths when you only need one result.

**2. shortestPath missing a minimum hop bound**
`shortestPath((a)-[*]->(b))` where `a` and `b` are the same node returns a zero-length path. Add `WHERE a <> b` or use `[*1..]` to enforce at least one hop.

**3. Confusing MATCH and OPTIONAL MATCH cardinality**
When `OPTIONAL MATCH` produces multiple rows (a patient with 3 prescriptions), subsequent MATCH clauses multiply results. Use `WITH` and aggregation to collapse before chaining further MATCH clauses.

**4. Undirected MATCH returning duplicate pairs**
`MATCH (a:Person)--(b:Person)` returns both `(Alice, Bob)` and `(Bob, Alice)`. Filter with `WHERE id(a) < id(b)` or use directional relationships to avoid duplicates.

**5. Using variable-length paths on unlabelled relationships**
`MATCH (a)-[*1..3]-(b)` traverses ALL relationship types. On a dense graph with many relationship types, this can match unexpected paths. Always specify the relationship type in variable-length patterns: `(a)-[:KNOWS*1..3]-(b)`.

**6. nodes(path) and relationships(path) only work on named paths**
`MATCH p = (a)-[*]->(b) RETURN nodes(p)` works. `MATCH (a)-[*]->(b) RETURN nodes(p)` fails — `p` is unbound. Always assign a path variable: `MATCH p = pattern`.


---
*sql_methods_library - Samantha McGarrigle*